# 03 · Practice Exercises

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samaksh97/grpo-then-and-now/blob/main/notebook/03_exercises.ipynb)

**GRPO, Then and Now** · **15–45 minutes** (Exercises 1 and 6 are core, 2–5 optional) · CPU only, runs in about 1 second

Each exercise has a starter cell with `TODO`s. Reference solutions are in `../solutions/exercise_solutions.py` (run `python solutions/exercise_solutions.py` from the repository root); Exercise 1 also has its answer in a collapsible cell.

**You need:** `01_grpo_core.ipynb` and `02_failure_lab.ipynb`. The setup cell below provides every function the starters use.

In [1]:
# Setup: the names the starter cells use
import sys, pathlib, subprocess
ROOT = next((p for p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent) if (p / "grpo_lab").exists()), None)
if ROOT is None:   # e.g. opened in Colab: fetch the repository (code, data and figures) first
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/samaksh97/grpo-then-and-now"], check=True)
    ROOT = pathlib.Path("grpo-then-and-now").resolve()
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from grpo_lab import data, envs
from grpo_lab.objectives import softmax, entropy, bandit_grpo_step, bandit_grpo_step_dynamic
from grpo_lab.experiments import train_sequence_grpo

cfg = envs.Config()
BANDIT_CANDIDATES = data.matmul_task()["candidates"]
verifier = envs.BanditVerifier(BANDIT_CANDIDATES, correct_action=cfg.bandit_correct_action)

## `03_exercises.ipynb`: Practice Exercises *(Exercises 1 and 6 core · 15+ min)*

Complete each learner cell before consulting `../solutions/exercise_solutions.py`, the authoritative reference implementation for all six exercises.

### Exercise 1: Compute Advantages by Hand (Analytical)

A group of 6 candidate responses receives rewards $R = [1, 0, 1, 1, 0, 0]$ from the verifier.

1. Compute the empirical mean $\bar{R}$ and standard deviation $\sigma(R)$.
2. Compute the group-relative advantage $A_i = \frac{R_i - \bar{R}}{\sigma + \epsilon}$ for each response.
3. Which candidates receive positive advantage (reinforced) and which receive negative advantage (suppressed)?

Complete the learner cell below before opening the numerical answer.

In [2]:
# Exercise 1: Your solution
rewards_ex1 = np.array([1, 0, 1, 1, 0, 0], dtype=float)

# TODO: compute mean, std, advantages
# mean_r = ...
# std_r = ...
# advantages_ex1 = ...

# Uncomment to check:
# print(f"Mean: {mean_r:.3f}")
# print(f"Std:  {std_r:.3f}")
# for i, (r, a) in enumerate(zip(rewards_ex1, advantages_ex1)):
#     print(f"  Response {i}: r={r:.0f}, A={a:+.3f}")

<details>
<summary><b>Exercise 1 numerical answer</b></summary>

$\bar R=0.500$, $\sigma=0.500$, and the advantages are
$[+1,-1,+1,+1,-1,-1]$ (up to the small numerical $\epsilon$). Successful responses are reinforced, failed responses are suppressed, and the advantages sum to zero.

See `../solutions/exercise_solutions.py::solve_exercise_1` for the complete reference implementation.
</details>

### Exercise 2: Find the Group-Size Trade-off

Using the verifier-based `bandit_grpo_step`, sweep $G\in\{4,8,16,32,64\}$ with seed 42. For each $G$, record the first pre-update step at which $P(\mathrm{optimal})\ge0.90$, the zero-gradient fraction over 300 requested steps, and generated responses ($300G$). Plot steps-to-threshold and zero-gradient fraction, then interpret quality together with rollout cost.

Complete the starter below. The full reference is `../solutions/exercise_solutions.py::solve_exercise_2`.

In [3]:
# Exercise 2: learner starter
np.random.seed(42)
group_sizes_ex2 = [4, 8, 16, 32, 64]
results_ex2 = {}

# TODO: for each G, reset the logits, run 300 requested updates, and record:
# - first step where P(optimal) >= 0.90
# - fraction of zero-gradient updates
# - total generated responses (= 300 * G)
# TODO: plot steps-to-threshold and zero-gradient fraction against G.

# Optional checks after implementation:
# assert set(results_ex2) == set(group_sizes_ex2)
# assert all(result['generated_responses'] == 300 * G for G, result in results_ex2.items())

### Exercise 3: Dynamic Sampling with Matched Requested Updates

Compare vanilla and Dynamic Sampling with the same seed, initial logits, group size, and 100 requested optimizer steps. Report final target probability, residual zero-gradient updates, and total generated responses. With at most five attempts, Dynamic Sampling **reduces** rather than guarantees elimination of uninformative updates.

Complete the starter below. The full reference is `../solutions/exercise_solutions.py::solve_exercise_3`.

In [4]:
# Exercise 3: learner starter
exercise3_results = {}

# TODO: run both bandit_grpo_step and bandit_grpo_step_dynamic from identical
# initial logits and seeds for 100 requested updates.
# TODO: record p_correct, zero_updates, and generated_responses for each method.
# TODO: explain the signal-versus-generation-cost trade-off.

# Optional checks after implementation:
# assert exercise3_results['Dynamic']['generated_responses'] >= exercise3_results['Vanilla']['generated_responses']

### Exercise 4: Run the Canonical Clip-Higher Comparison

Run the fixed-rollout sequence toy with $(\epsilon_{low},\epsilon_{high})$ equal to $(0.20,0.20)$, $(0.20,0.28)$, and $(0.20,0.50)$. Plot actual maximum ratios and clipped fractions. Explain why clip bounds cannot matter on the first optimizer epoch, when current and behavior policies are identical.

Complete the starter below. The full reference is `../solutions/exercise_solutions.py::solve_exercise_4`.

In [5]:
# Exercise 4: learner starter
configs_ex4 = [
    ('Symmetric', 0.20, 0.20),
    ('DAPO', 0.20, 0.28),
    ('Wide toy', 0.20, 0.50),
]
results_ex4 = {}

# TODO: call train_sequence_grpo for each configuration with a matched seed.
# TODO: collect max_ratio and clip_fraction, then plot both diagnostics.
# TODO: explain why epoch 1 starts with ratios equal to 1.

### Exercise 5: Learn a Toy Verifier Exploit

In an explicitly toy candidate-policy experiment, make a vulnerable verifier reward any non-empty sequence ending in `END`. Train a categorical policy and show that a meaningless `[..., END]` candidate gains probability. Then use a behavior-based verifier that interprets valid operations, checks the final property, accepts multiple equivalent implementations, and safely rejects empty or malformed sequences.

`02_failure_lab.ipynb` (Appendix A) shows a related demonstration. Complete the starter below. The full reference is `../solutions/exercise_solutions.py::solve_exercise_5`.

In [6]:
# Exercise 5: learner starter
END = 9
candidates_ex5 = [[], [0, 0, END], [2, 1, END], [1, 1, END], [2, 2, END]]

# TODO: implement vulnerable_verifier(seq).
# TODO: implement robust_verifier(seq, target_value=3) using behavioral checks.
# TODO: train the candidate policy with each verifier and compare exploit probability.

# Optional checks after implementation:
# assert robust_verifier([]) == 0
# assert robust_verifier([0, 0, END]) == 0
# assert robust_verifier([2, 1, END]) == robust_verifier([1, 1, END]) == 1

### Exercise 6: Diagnostic Scenarios

For each scenario below, name the failure and the most relevant response:

1. Most groups on a hard task contain identical rewards.
2. Positive-advantage ratios repeatedly hit the upper clip bound.
3. Correct responses range from 8 to 200 tokens under sample-level normalization.
4. Responses enter the final cache region or hit the truncation boundary.
5. Training reward is perfect, but programs exploit a superficial verifier and fail randomized held-out tests.

Complete the mapping below. The full reference is `../solutions/exercise_solutions.py::solve_exercise_6`.

In [7]:
# Exercise 6: learner starter
scenarios_ex6 = [
    'identical rewards on a hard task',
    'positive ratios hit the upper clip bound',
    'correct responses have very different lengths',
    'responses reach the cache or truncation region',
    'perfect training reward but held-out verifier failure',
]

# TODO: map each scenario to a diagnosis and response.
# answers_ex6 = {
#     scenarios_ex6[0]: ('...', '...'),
# }